# 03 - Final data preparation (all data, no graphs)

**Who worked on this:**Mohammed AbuQamar 

This is the "recipe" the assignment asks for: every cleaning step we decided on in `02_clean_eda`, and nothing else. No plots, no experiments. It goes from the raw parquet files (01) to clean parquet files that the models use.

| step | function in `src/cleaning.py` | why (see 02) |
|---|---|---|
| 1 | `set_types` | NA placeholders, timestamps in 4 formats, numbers, legacy labels -> current labels |
| 2 | `drop_duplicates` | `ride_id` unique; legacy: same bike + times + stations |
| 3 | `handle_missing` | fill structural NAs (`no_station`, `unknown`), drop trips without times/coordinates |
| 4 | `add_trip_metrics` | `duration_min`, `distance_km`, `speed_kmh` |
| 5 | `filter_duration` | keep 1 min - 24 h |
| 6 | `filter_location` | keep NYC + Jersey City box |
| 7 | `filter_speed` | drop > 40 km/h |
| 8 | `finalize` | keep needed columns, fixed categoricals |

**Why the code lives in `src/`:** the automated retraining pipeline (GitHub Actions) runs plain Python, not notebooks. Keeping one copy of the cleaning means 02, 03 and the pipeline can never drift apart.

**How it scales:** each raw file is cleaned in batches of 1M rows, so memory stays small. Finished files are skipped, so you can stop and resume. Expect very roughly 30-60 minutes for everything.

In [1]:
import sys
sys.path.append(".")
import pandas as pd
import duckdb
from src.cleaning import prepare_all, RAW_DIR, CLEAN_DIR

## 1. Clean every file

In [2]:
report = prepare_all()        # prepare_all(rebuild=True) to redo everything after changing a rule
report.tail()

[1/424] 201306-citibike-tripdata.parquet: 577,703 -> 542,917 rows
[2/424] 201306-citibike-tripdata_1.parquet: 577,703 -> 542,917 rows
[3/424] 201307-citibike-tripdata.parquet: 843,416 -> 824,652 rows
[4/424] 201307-citibike-tripdata_1.parquet: 843,416 -> 824,652 rows
[5/424] 201308-citibike-tripdata.parquet: 1,001,958 -> 984,167 rows
[6/424] 201308-citibike-tripdata_1.parquet: 1,000,000 -> 982,230 rows
[7/424] 201308-citibike-tripdata_2.parquet: 1,958 -> 1,937 rows
[8/424] 201309-citibike-tripdata.parquet: 1,034,359 -> 1,019,098 rows
[9/424] 201309-citibike-tripdata_1.parquet: 1,000,000 -> 985,221 rows
[10/424] 201309-citibike-tripdata_2.parquet: 34,359 -> 33,877 rows
[11/424] 201310-citibike-tripdata.parquet: 1,037,712 -> 1,022,924 rows
[12/424] 201310-citibike-tripdata_1.parquet: 1,000,000 -> 985,751 rows
[13/424] 201310-citibike-tripdata_2.parquet: 37,712 -> 37,173 rows
[14/424] 201311-citibike-tripdata.parquet: 675,774 -> 667,105 rows
[15/424] 201311-citibike-tripdata_1.parquet: 67

,file,raw,set_types,drop_duplicates,handle_missing,add_trip_metrics,filter_duration,filter_location,filter_speed,finalize
419,202609/202609-citibike-tripdata_1.parquet,1000000,1000000,1000000,997312,997312,997309,997309,997309,997309
420,202609/202609-citibike-tripdata_2.parquet,1000000,1000000,1000000,994473,994473,994456,994456,994456,994456
421,202609/202609-citibike-tripdata_3.parquet,1000000,1000000,1000000,994401,994401,994390,994390,994390,994390
422,202609/202609-citibike-tripdata_4.parquet,1000000,1000000,1000000,997406,997406,997399,997399,997398,997398
423,202609/202609-citibike-tripdata_5.parquet,982855,982855,982855,982121,982121,982118,982118,982118,982118


## 2. Cleaning report: what did each step remove?
This also gives the real duplicate counts, which the sample in 02 could not show.

In [3]:
steps = [c for c in report.columns if c != "file"]
totals = report[steps].sum()
summary = pd.DataFrame({
    "rows left": totals,
    "removed by step": (-totals.diff()).fillna(0).astype(int),
})
summary["% of raw removed"] = (summary["removed by step"] / totals["raw"] * 100).round(3)
display(summary)
print(f"Kept {totals.iloc[-1] / totals['raw']:.2%} of {totals['raw']:,} raw trips")

,rows left,removed by step,% of raw removed
raw,351963720,0,0.000
set_types,351963720,0,0.000
drop_duplicates,351963715,5,0.000
handle_missing,351268686,695029,0.197
add_trip_metrics,351268686,0,0.000
filter_duration,351210059,58627,0.017
filter_location,351209373,686,0.000
filter_speed,351002952,206421,0.059
finalize,351002952,0,0.000


Kept 99.73% of 351,963,720 raw trips


In [4]:
report["year"] = report["file"].str[:4]
by_year = report.groupby("year")[["raw", "finalize"]].sum()
by_year["% kept"] = (by_year["finalize"] / by_year["raw"] * 100).round(2)
by_year

,raw,finalize,% kept
year,,,
2013,11229776,10998840,97.94
2014,8081216,8071246,99.88
2015,9937969,9934862,99.97
2016,13845655,13841805,99.97
2017,16364657,16360263,99.97
2018,35096678,35087116,99.97
2019,20551697,20545563,99.97
2020,19562314,19502869,99.70
2021,27130122,27059254,99.74


If one year keeps a much lower share than the others, look at that year in 02 again: it usually means a format problem, not bad trips.

## 3. Final check on the clean data
One schema for all files, no NAs in the key columns, and a sensible date range.

In [5]:
con = duckdb.connect()
src = f"read_parquet('{CLEAN_DIR.as_posix()}/**/*.parquet')"
display(con.execute(f"DESCRIBE SELECT * FROM {src}").df()[["column_name", "column_type"]])
display(con.execute(f'''
    SELECT count(*) AS trips, min(started_at) AS first_trip, max(started_at) AS last_trip,
           count(*) FILTER (WHERE started_at IS NULL OR start_lat IS NULL OR start_station_name IS NULL) AS rows_with_na,
           round(min(duration_min), 1) AS min_duration, round(max(duration_min), 1) AS max_duration
    FROM {src}''').df())

,column_name,column_type
0,started_at,TIMESTAMP
1,ended_at,TIMESTAMP
2,start_station_id,VARCHAR
3,start_station_name,VARCHAR
4,end_station_id,VARCHAR
5,end_station_name,VARCHAR
6,start_lat,DOUBLE
7,start_lng,DOUBLE
8,end_lat,DOUBLE
9,end_lng,DOUBLE


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,trips,first_trip,last_trip,rows_with_na,min_duration,max_duration
0,351002952,2013-06-01 00:00:01,2026-09-30 23:58:16.821,0,1.0,1440.0


**Disk tip:** once this has run, `citibike_data/raw/` is only needed again if you change a cleaning rule. If disk space is tight you can delete it, but keep `citibike_data/raw_sample.parquet` so 02 still runs. Re-downloading means running 01 again.